## Imports

In [1]:
import pandas as pd 
import numpy as np

# Splits the dataset into training and testing sets.
from sklearn.model_selection import train_test_split
# Finds the best hyperparameters using cross-validation.
from sklearn.model_selection import GridSearchCV

# Scales features to a specified range, usually 0 to 1.
from sklearn.preprocessing import MinMaxScaler

# Classifies data based on the nearest neighbors.
from sklearn.neighbors import KNeighborsClassifier

# Calculates the proportion of correct predictions.
from sklearn.metrics import accuracy_score

# Shows correct and incorrect predictions for each class.
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt 
import seaborn as sns 


data = pd.read_csv("titanic.csv")

In [2]:
data.info()
print(f"####################\n")
print(data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     object 
 11  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 39.3+ KB
####################

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked     

#Data Cleaning and Features Engineering

In [3]:
def preprocess_data(df):
    df.drop(columns=["PassengerId","Name","Ticket","Cabin"], inplace=True)
    df["Embarked"].fillna("S", inplace=True)
    df.drop(columns = ["Embarked"], inplace = True)
    
    
    fill_missing_ages(df)
    
    #convert Gender
    df["Sex"] = df["Sex"].map({'male':1, 'female':0})
    
    #Feature Engineering
    df['FamilySize'] = df["SibSp"] + df["Parch"]
    df['IsAlone'] = np.where(df["FamilySize"] == 0, 1, 0)
    df["FareBin"] = pd.qcut(df["Fare"], 4, labels = False)
    df["AgeBin"] = pd.cut(df["Age"], bins=[0, 12, 20, 40, 60, np.inf], labels = False)
    
    return df

# Fill missing ages
def fill_missing_ages(df):
    age_fill_map = {}
    for pclass in df["Pclass"].unique():  # class: 1,2,3
        if pclass not in age_fill_map:
            age_fill_map[pclass] = df[df["Pclass"] == pclass]["Age"].median()
    

    df["Age"] = df.apply(lambda row: age_fill_map[row["Pclass"]] if pd.isnull(row["Age"]) else row["Age"], axis = 1)
    
data = preprocess_data(data)

C:\Users\chath\AppData\Local\Temp\ipykernel_27456\680669580.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Embarked"].fillna("S", inplace=True)


In [4]:
#create Features / Target Variables (Make Flashcards)

X = data.drop(columns = ["Survived"])
y = data['Survived']

#flashcards
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

In [5]:
# ML preprocessing
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Hyperparameter Tuning - KNN

In [6]:
def tune_model(X_train, y_train):
    param_grid = {
        "n_neighbors" : range(1,21),
        "metrics" : {"euclidean", "manhatton", "minkowski"},
        "weights" : {"uniform", "distance"}
    }
    
    model = KNeighborsClassifier()
    grid_search = GridSearchCV(model, param_grid, cv=5, n_jobs = -1)
    grid_search.fit(X_train, y_train)
    return grid_search.best_estimator_

In [ ]:
best_model = tune_model(X_train, y_train) #this is the model